In [ ]:
import ibis
from utils.f_0_dirs import get_data_dirs

table_name_fixed = "working_fixed"
table_name_yearly = "working_yearly"
table_name_peers = "working_yearly_peers"
column_peers = ["tfp", "gva1", "total_assets", "employees"]
spatial_groups = ["pc8", "pc4", "ttwa"]

dirs = get_data_dirs(segment="calculations")
con = ibis.duckdb.connect(str(dirs.db_path))
con.raw_sql("INSTALL spatial; LOAD spatial;")

table_fixed = con.table(table_name_fixed)
table_yearly = con.table(table_name_yearly)
table_joined = (
    table_yearly
    .distinct(on=['registered_number', 'year'])
    # Positive values only for the peer calculations. Generate filter condition that >0 for every column_peers
    .filter(ibis.and_(*[ibis._[c] > 0 for c in column_peers]))
    .left_join(
        table_fixed,
        "registered_number"
    )
)
columns_yearly_str = ", ".join(table_yearly.columns)

table_name_view = "spatial_panel_view"
con.create_view(table_name_view, table_joined, overwrite=True)

agg_sql_str = ""
calc_sql_str = ""
for i, g in enumerate(spatial_groups):

    for c in column_peers:
    
        prior_group = f"count_{c}_{spatial_groups[i-1]}" if i > 0 else '1'
        donut_str = '_d' if i > 0 else ''

        agg_sql_str += f"""
        SUM({c}) OVER (PARTITION BY {g}, year) AS sum_{c}_{g},
        COUNT({c}) OVER (PARTITION BY {g}, year) AS count_{c}_{g},
        """

        calc_sql_str += f"""
        (sum_{c}_{g} - {c}) / NULLIF(count_{c}_{g} - {prior_group}, 0) AS {c}_{g}{donut_str},
        """

# ==========================================
# 2. PARTITION, AGGREGATE, BROADCAST, TRANSFORM
# ==========================================
# We use 3 concentric rings. 
# Ring 1: extracted_postcode (Innermost)
# Ring 2: pc4 (Middle Donut)
# Ring 3: ttwa (Outer Donut)
window_query_body = f"""
WITH GroupAggregates AS (
    SELECT 
        {columns_yearly_str},
        {",\n".join(column_peers)},        
        {agg_sql_str}
    FROM 
        {table_name_view}
)
SELECT 
    registered_number,
    year,    
    {calc_sql_str}
    
FROM 
    GroupAggregates
"""

# 2. Write new table directly in the raw sql
create_table_ddl = (
    f"CREATE OR REPLACE TABLE {table_name_peers} AS ({window_query_body});"
)

# 3. Execute directly on DuckDB connection
con.raw_sql(create_table_ddl)

# 4. Bind the newly materialized physical table back to Ibis
table_peers = con.table(table_name_peers)
# View the schema to verify the 3 new mutually exclusive spatial donut columns
print(table_peers.schema())
# Print the head of the resulting table to verify the calculations
display(table_peers.sample(0.0001).execute())

ibis.Schema {
  registered_number    string
  year                 int64
  tfp_pc8              float64
  gva1_pc8             float64
  total_assets_pc8     float64
  employees_pc8        float64
  tfp_pc4_d            float64
  gva1_pc4_d           float64
  total_assets_pc4_d   float64
  employees_pc4_d      float64
  tfp_ttwa_d           float64
  gva1_ttwa_d          float64
  total_assets_ttwa_d  float64
  employees_ttwa_d     float64
}


,registered_number,year,tfp_pc8,gva1_pc8,total_assets_pc8,employees_pc8,tfp_pc4_d,gva1_pc4_d,total_assets_pc4_d,employees_pc4_d,tfp_ttwa_d,gva1_ttwa_d,total_assets_ttwa_d,employees_ttwa_d
0,08709411,2023,NaN,NaN,NaN,NaN,2.876256,6492.406734,2.393534e+04,119.924528,3.360557,13103.450266,7.330292e+04,230.955882
1,05192763,2007,2.988204,10703.874264,33574.441948,235.846154,5.328308,215375.013610,1.153271e+06,1397.681818,3.225666,75271.231128,7.390750e+05,813.583880
2,06192910,2011,3.619218,16947.957676,37661.315267,219.300000,3.576484,54498.566609,2.449917e+05,839.836207,3.206451,100949.741301,2.301179e+06,865.292993
3,SC092520,2021,NaN,NaN,NaN,NaN,3.015794,1313.959584,8.187752e+02,38.000000,2.952515,21748.145286,2.173160e+05,315.174619
4,04136968,2007,2.789178,33870.303119,130414.277870,740.529412,3.463084,16242.386507,7.070578e+04,259.529412,3.155468,15151.915282,6.285926e+04,326.303514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,04231521,2020,3.248076,22495.014966,50060.983209,731.333333,3.218896,8486.671533,1.865960e+04,307.911765,3.357810,13964.532408,5.731414e+04,262.135371
110,02214956,2019,NaN,NaN,NaN,NaN,2.777234,1972.002405,4.076715e+03,43.000000,3.201318,88278.181724,1.853400e+06,795.243337
111,05735653,2022,NaN,NaN,NaN,NaN,2.974986,4589.644387,8.374156e+03,105.933333,3.103540,36005.521877,1.649380e+05,808.130693
112,03618688,2014,3.059305,16140.884596,130300.858920,212.950000,3.857429,33398.546199,1.633221e+05,572.298507,4.931910,30288.534935,1.271811e+05,541.967480


In [ ]:
from ibis import _

const_d = 1000  # Example constant to divide by for exponential decay

# Industry codes processing
t_fixed_ind = (
    t_fixed
    .select(["registered_number", "sic6"])
    .mutate(sic2=_.sic6.substr(0, 2))
    .distinct()
)
t_panel_skinny = (
    t_panel
    .select(["registered_number", "year", "tfp"])
    .left_join(t_fixed_ind, "registered_number")
)

# 1a. Distance setup (based on target firms)
t_distance_short = (
    t_distance
    .inner_join(t_panel_skinny, _.firm_i == t_panel_skinny.registered_number)
    .mutate(
        d1 = 1 / (_.distance_meters + 1),
        d2 = 1 / (_.distance_meters + 1) ** 2,
        d3 = (-_.distance_meters / const_d).exp()
    )
)

# 1b. Peer panel setup
t_panel_peer = t_panel_skinny.select(
    firm_j="registered_number",
    peer_year="year",
    peer_tfp="tfp",
    peer_sic2="sic2",
    peer_sic6="sic6"
)

# 2. Join peer data, matching on firm ID and year
# For every distance record
t_distance_raw = (
    t_distance_short
    .left_join(
        t_panel_peer, 
        (_.firm_j == t_panel_peer.firm_j) & (_.year == t_panel_peer.peer_year)
    )
    .filter(_.peer_tfp.notnull())
)
t_distance_wd = (
    t_distance_raw
    .mutate(
        tfp_w1=_.peer_tfp * _.d1,
        tfp_w2=_.peer_tfp * _.d2,
        tfp_w3=_.peer_tfp * _.d3
    )
    .group_by("firm_i", "year")
    .aggregate(
        tfp_wav1=_.tfp_w1.sum() / _.d1.sum(),
        tfp_wav2=_.tfp_w2.sum() / _.d2.sum(),
        tfp_wav3=_.tfp_w3.sum() / _.d3.sum(),
        nb_peers=_.peer_tfp.count()
    )
)
t_distance_wd_2i = (
    t_distance_raw
    .filter(_.firm_)
)

t_panel_tfpd = (
    t_panel
    .left_join(
        t_distance_tfp, 
        (_.registered_number == t_distance_tfp.firm_i) & (_.year == t_distance_tfp.year),
        rname="{name}_dist"
    )
    .drop(["firm_i", "year_dist"])
    # .order_by(ibis.random())
)

df_peers = (
    t_panel_tfpd
    .select(['registered_number', 'year', 'tfp', 'tfp_wav1', 'tfp_wav2', 'tfp_wav3', 'nb_peers'])
)

In [ ]:
import ibis
from ibis import _
from utils.f_0_dirs import get_data_dirs

# Calculate group-based spatial lag using a leave-one-out or donut approach
def apply_group_weight(t_panel, input_col, group_col, out_col, inner_sum_col=None, inner_count_col=None):
    
    # 1. Calculate partition aggregates using fast window functions
    sum_col = t_panel[input_col].sum().over(group_by=[t_panel[group_col], t_panel.year])
    count_col = t_panel[input_col].count().over(group_by=[t_panel[group_col], t_panel.year])
    
    # 2. Apply zero-diagonal & donut logic
    if inner_sum_col is not None and inner_count_col is not None:
        numerator = sum_col - t_panel[inner_sum_col]
        denominator = count_col - t_panel[inner_count_col]
    else:
        numerator = sum_col - t_panel[input_col]
        denominator = count_col - 1
        
    # 3. Row-normalize via division, coercing to avoid ZeroDivisionError
    t_panel_mutated = t_panel.mutate(**{
        out_col: ibis.ifelse(denominator > 0, numerator / denominator, ibis.NA),
        f"{out_col}_sum": sum_col,      
        f"{out_col}_count": count_col   
    })
    
    return t_panel_mutated


# Calculate network-based distance decay spatial lag
def apply_network_weight(t_panel, t_distance, input_col, weight_col, out_col):
    
    # 1. Isolate target and peer data to prevent temporal cartesian products
    t_target = t_panel.select(firm_i="registered_number", target_year="year")
    t_peer = t_panel.select(firm_j="registered_number", peer_year="year", peer_val=input_col)

    # 2. Join distances to valid peers operating in the exact same year
    t_distance_weighted = (
        t_distance
        .inner_join(t_target, t_distance.firm_i == _.firm_i)
        .inner_join(t_peer, (t_distance.firm_j == _.firm_j) & (_.target_year == _.peer_year))
        .filter(_.peer_val.notnull()) 
        .mutate(weighted_val=_.peer_val * _[weight_col])
        .group_by("firm_i", "target_year")
        .aggregate(**{
            # Implicit row-normalization: divide sum product by sum of valid weights
            out_col: _.weighted_val.sum() / _[weight_col].sum()
        })
    )

    # 3. Safely join the calculated spatial lag back to the main panel
    t_panel_mutated = (
        t_panel
        .left_join(
            t_distance_weighted, 
            (t_panel.registered_number == _.firm_i) & (t_panel.year == _.target_year)
        )
        .drop(["firm_i", "target_year"])
    )
    
    return t_panel_mutated


# Iterative Application Example
variables_to_lag = ["gva1", "total_assets", "employees"]
panel_name = "working_yearly"
fixed_name = "working_fixed"
distance_name = "working_distance_ttwa_km"

# ---
dirs = get_data_dirs(segment="calculations")
con = ibis.duckdb.connect(dirs.db_path)
dirs = get_data_dirs(segment="calculations")

table_panel = (
    con.table(panel_name)
)
table_fixed = con.table(fixed_name)
table_distance = con.table(distance_name)
t_current = table_panel

for var in variables_to_lag:
    # A. Apply innermost group (Leave-one-out)
    t_current = apply_group_weight(t_current, var, "pc8", f"{var}_pc8")
    
    # B. Apply outer donut (Subtract inner group sum/count)
    t_current = apply_group_weight(
        t_current, var, "pc4", f"{var}_pc4_d", 
        inner_sum_col=f"{var}_pc8_sum", inner_count_col=f"{var}_pc8_count"
    )
    
    # C. Apply continuous network decay
    t_current = apply_network_weight(t_current, table_distance, var, "d1", f"{var}_d1")

print(t_current.schema())
print(t_current.sample(0.0001).execute())

IbisTypeError: Column 'pc8' is not found in table. Existing columns: 'registered_number', 'year', 'employees', 'fixed_total', 'total_assets', 'average_wage', 'gva1', 'gva2', 'gva1_per_worker', 'gva2_per_worker', 'tfp'.